In [ ]:
import math
class Value:
    def __init__(self, data, parents=()):
        self.data = data
        self.parents = parents
        self.grad = 0
        self._backward = lambda: None
    
    def __add__(self, val):
        if not isinstance(val, Value):
            val = Value(val)
        out = Value(self.data + val.data, parents=(self,val))

        def _backward():
            self.grad += 1 * out.grad
            val.grad += 1 * out.grad

        out._backward = _backward

        return out
    
    def __radd__(self, val):
        return self.__add__(val)
    
    def __sub__(self, val):
        if not isinstance(val, Value):
            val = Value(val)
        out = Value(self.data - val.data, parents=(self,val))

        def _backward():
            self.grad += 1 * out.grad
            val.grad += -1 * out.grad

        out._backward = _backward
        return out

    def __rsub__(self, val):
        return val + (-self)
        
    
    def __mul__(self, val):
        if not isinstance(val, Value):
            val = Value(val)
        out = Value(self.data * val.data, parents=(self, val))

        def _backward():
            self.grad += val.data * out.grad
            val.grad += self.data * out.grad

        out._backward = _backward
        return out
    
    def __rmul__(self, val):
        return self.__mul__(val)
    
    def __truediv__(self, val):
        if not isinstance(val, Value):
            val = Value(val)
        out = Value(self.data / val.data, parents=(self,val))
    

        def _backward():
            self.grad += (1 / val.data) * out.grad
            val.grad += (-self.data / (val.data**2)) * out.grad
        
        out._backward = _backward

        return out
        

    
    def __rtruediv__(self, val):
        return val * (self ** -1)

    def __pow__(self, val):
        if not isinstance(val, Value):
            out = Value(self.data ** val, parents = (self,))
            def _backward():
                self.grad +=  (val * self.data ** (val-1)) * out.grad
        else:
            out = Value(self.data ** val.data, parents=(self,val))
            def _backward():
                self.grad +=  (val.data * self.data ** (val.data-1)) * out.grad
                val.grad += math.log(self.data) * out.data * out.grad
        
        out._backward = _backward
        return out
        
        
    
    def __neg__(self):
        out = Value(self.data * -1, parents = (self,))
        def _backward():
                self.grad += -out.grad
        out._backward = _backward
        return out
    
    def relu(self):
        out = Value(max(0, self.data), parents = (self,))

        if out.data == 0:
            def _backward():
                self.grad += 0.0
        else:
            def _backward():
                self.grad += out.grad
        
        out._backward = _backward

        return out
    
    def backward(self):
        visited = set()
        topo = []

        def build_topo(val):
            if val not in visited:
                visited.add(val)
                for parent in val.parents:
                    build_topo(parent)
                topo.append(val)
        
        build_topo(self)
        self.grad = 1.0
        
        for val in reversed(topo):
            val._backward()